In [1]:
from pyspark.sql import SparkSession


# Initialize Spark Session


spark = SparkSession.builder \
.appName("IcebergLocalSetup") \
.config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
.config("spark.sql.catalog.local.type", "hadoop") \
.config("spark.sql.catalog.local.warehouse", "file:///C:/iceberg_data_warehouse/w1/") \
.config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
.config("spark.jars", "file:///C:/iceberg/iceberg-spark-runtime-4.0_2.13-1.10.0.jar") \
.getOrCreate()

In [2]:
import pyspark
print(pyspark.__version__)


4.0.1


In [3]:
 
# Create Iceberg table
spark.sql("""
    CREATE OR REPLACE TABLE local.db.people (
        id INT,
        name STRING,
        age INT
    )
    USING ICEBERG
""")

DataFrame[]

In [4]:
spark.sql("""
    ALTER TABLE local.db.people SET TBLPROPERTIES (
        'write.metadata.delete-after-commit.enabled' = 'false',
        'write.metadata.previous-versions-max' = '5',
        'read.change.data.feed' = 'true'
    )
""")


DataFrame[]

In [5]:
spark.sql("SHOW TBLPROPERTIES local.db.people").show(truncate=False)

+------------------------------------------+---------------+
|key                                       |value          |
+------------------------------------------+---------------+
|current-snapshot-id                       |none           |
|format                                    |iceberg/parquet|
|format-version                            |2              |
|read.change.data.feed                     |true           |
|write.metadata.delete-after-commit.enabled|false          |
|write.metadata.previous-versions-max      |5              |
|write.parquet.compression-codec           |zstd           |
+------------------------------------------+---------------+



In [6]:
# Insert two records in a table
spark.sql("""
    INSERT INTO local.db.people VALUES
    (1, 'Nootan', 28),
    (2, 'Neha', 31)
""")

DataFrame[]

In [7]:
# Read data
spark.sql("SELECT * FROM local.db.people").show()
 

+---+------+---+
| id|  name|age|
+---+------+---+
|  1|Nootan| 28|
|  2|  Neha| 31|
+---+------+---+



In [8]:

# adding a new column 'city' in a table
spark.sql("""
    ALTER TABLE local.db.people ADD COLUMN city STRING
""")


DataFrame[]

In [9]:
# describing table schema
spark.sql("DESCRIBE TABLE local.db.people").show(truncate=False)
 

+--------+---------+-------+
|col_name|data_type|comment|
+--------+---------+-------+
|id      |int      |NULL   |
|name    |string   |NULL   |
|age     |int      |NULL   |
|city    |string   |NULL   |
+--------+---------+-------+



In [10]:
# inserting new records 
spark.sql("""
    INSERT INTO local.db.people VALUES
    (3, 'Rohit', 28, 'Mumbai'),
    (4, 'Virat', 31, 'Pune')
""")
 
# reading data from table
spark.sql("SELECT * FROM local.db.people").show()

+---+------+---+------+
| id|  name|age|  city|
+---+------+---+------+
|  1|Nootan| 28|  NULL|
|  2|  Neha| 31|  NULL|
|  3| Rohit| 28|Mumbai|
|  4| Virat| 31|  Pune|
+---+------+---+------+



In [11]:
# track schema versions
spark.sql("SELECT * FROM local.db.people.snapshots").show(truncate=False)
 

+-----------------------+-------------------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                             |summary                                                                                                                                         

In [12]:
spark.sql("""
    UPDATE local.db.people
    SET city = 'Ahmedabad'
    WHERE name = 'Nootan'
""")
 
# reading data from table
spark.sql("SELECT * FROM local.db.people").show()
 
spark.sql("SELECT * FROM local.db.people.snapshots").show()


+---+------+---+---------+
| id|  name|age|     city|
+---+------+---+---------+
|  3| Rohit| 28|   Mumbai|
|  4| Virat| 31|     Pune|
|  1|Nootan| 28|Ahmedabad|
|  2|  Neha| 31|     NULL|
+---+------+---+---------+

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2025-11-17 16:10:...|5383781062247318359|               NULL|   append|file:/C:/iceberg_...|{spark.app.id -> ...|
|2025-11-17 16:11:...|7820283151589575266|5383781062247318359|   append|file:/C:/iceberg_...|{spark.app.id -> ...|
|2025-11-17 16:12:...|5989816188880001704|7820283151589575266|overwrite|file:/C:/iceberg_...|{spark.app.id -> ...|
+--------------------+-------------------+-------------------+---------+--------------------+

In [13]:
spark.sql("""
    DELETE FROM local.db.people
    WHERE age > 30
""")
spark.sql("SELECT * FROM local.db.people").show()
 
spark.sql("SELECT * FROM local.db.people.snapshots").show()
 

+---+------+---+---------+
| id|  name|age|     city|
+---+------+---+---------+
|  1|Nootan| 28|Ahmedabad|
|  3| Rohit| 28|   Mumbai|
+---+------+---+---------+

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2025-11-17 16:10:...|5383781062247318359|               NULL|   append|file:/C:/iceberg_...|{spark.app.id -> ...|
|2025-11-17 16:11:...|7820283151589575266|5383781062247318359|   append|file:/C:/iceberg_...|{spark.app.id -> ...|
|2025-11-17 16:12:...|5989816188880001704|7820283151589575266|overwrite|file:/C:/iceberg_...|{spark.app.id -> ...|
|2025-11-17 16:14:...|7458428039780159248|5989816188880001704|   delete|file:/C:/iceberg_...|{spark.app.id -> ...|
+--------------------+----------

In [16]:
spark.sql("SELECT committed_at,snapshot_id,operation FROM local.db.people.snapshots").show(truncate = False)
 
df = spark.read.option("snapshot-id", "5989816188880001704").table("local.db.people")
df.show()
 

+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2025-11-17 16:10:02.171|5383781062247318359|append   |
|2025-11-17 16:11:42.766|7820283151589575266|append   |
|2025-11-17 16:12:58.76 |5989816188880001704|overwrite|
|2025-11-17 16:14:08.82 |7458428039780159248|delete   |
+-----------------------+-------------------+---------+

+---+------+---+---------+
| id|  name|age|     city|
+---+------+---+---------+
|  3| Rohit| 28|   Mumbai|
|  2|  Neha| 31|     NULL|
|  4| Virat| 31|     Pune|
|  1|Nootan| 28|Ahmedabad|
+---+------+---+---------+



In [17]:
from datetime import datetime

timestamp_str = "2025-11-17 16:12:30.077"
timestamp_ms = int(datetime.strptime(timestamp_str, "%Y-%m-%d %H:%M:%S.%f").timestamp() * 1000)
print(timestamp_ms)


1763376150077


In [18]:
df = spark.read.option("as-of-timestamp", "1763374050077").table("local.db.people")
df.show()

IllegalArgumentException: Cannot find a snapshot older than 2025-11-17T10:07:30.077+00:00

In [21]:
snapshot_id = 7458428039780159248
 
spark.sql(f"""
    CALL local.system.rollback_to_snapshot('db.people', {snapshot_id})
""")
spark.sql("SELECT * FROM local.db.people").show()
spark.sql("SELECT committed_at,snapshot_id,operation FROM local.db.people.snapshots").show(truncate = False)

Py4JJavaError: An error occurred while calling o38.sql.
: org.apache.iceberg.exceptions.ValidationException: Cannot roll back to snapshot, not an ancestor of the current state: 7458428039780159248
	at org.apache.iceberg.exceptions.ValidationException.check(ValidationException.java:49)
	at org.apache.iceberg.SetSnapshotOperation.rollbackTo(SetSnapshotOperation.java:84)
	at org.apache.iceberg.SnapshotManager.rollbackTo(SnapshotManager.java:69)
	at org.apache.iceberg.spark.procedures.RollbackToSnapshotProcedure.lambda$call$0(RollbackToSnapshotProcedure.java:93)
	at org.apache.iceberg.spark.procedures.BaseProcedure.execute(BaseProcedure.java:132)
	at org.apache.iceberg.spark.procedures.BaseProcedure.modifyIcebergTable(BaseProcedure.java:113)
	at org.apache.iceberg.spark.procedures.RollbackToSnapshotProcedure.call(RollbackToSnapshotProcedure.java:88)
	at org.apache.spark.sql.catalyst.analysis.InvokeProcedures.org$apache$spark$sql$catalyst$analysis$InvokeProcedures$$invoke(InvokeProc

In [23]:
spark.sql("SELECT * FROM local.db.people.manifests").show(truncate=False)

+-------+--------------------------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+-------------------+
|content|path                                                                                              |length|partition_spec_id|added_snapshot_id  |added_data_files_count|existing_data_files_count|deleted_data_files_count|added_delete_files_count|existing_delete_files_count|deleted_delete_files_count|partition_summaries|
+-------+--------------------------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+-------------------+
|0      |file:/C

In [24]:
spark.sql("SELECT * FROM local.db.people.changes").show(truncate=False)

+---+------+---+---------+------------+---------------+-------------------+
|id |name  |age|city     |_change_type|_change_ordinal|_commit_snapshot_id|
+---+------+---+---------+------------+---------------+-------------------+
|1  |Nootan|28 |Ahmedabad|INSERT      |2              |5989816188880001704|
|3  |Rohit |28 |Mumbai   |INSERT      |1              |7820283151589575266|
|4  |Virat |31 |Pune     |INSERT      |1              |7820283151589575266|
|1  |Nootan|28 |NULL     |DELETE      |2              |5989816188880001704|
|1  |Nootan|28 |NULL     |INSERT      |0              |5383781062247318359|
|2  |Neha  |31 |NULL     |INSERT      |0              |5383781062247318359|
+---+------+---+---------+------------+---------------+-------------------+



In [25]:
spark.sql("""
SELECT * FROM local.db.people.changes
WHERE id = 1
ORDER BY _commit_snapshot_id
""").show()
 
spark.sql("""
SELECT * FROM local.db.people VERSION AS OF 7820283151589575266
""").show()
 


+---+------+---+---------+------------+---------------+-------------------+
| id|  name|age|     city|_change_type|_change_ordinal|_commit_snapshot_id|
+---+------+---+---------+------------+---------------+-------------------+
|  1|Nootan| 28|     NULL|      INSERT|              0|5383781062247318359|
|  1|Nootan| 28|     NULL|      DELETE|              2|5989816188880001704|
|  1|Nootan| 28|Ahmedabad|      INSERT|              2|5989816188880001704|
+---+------+---+---------+------------+---------------+-------------------+

+---+------+---+------+
| id|  name|age|  city|
+---+------+---+------+
|  3| Rohit| 28|Mumbai|
|  4| Virat| 31|  Pune|
|  1|Nootan| 28|  NULL|
|  2|  Neha| 31|  NULL|
+---+------+---+------+



In [26]:

spark.read.table("local.db.people.changes").printSchema()
 
spark.sql("SELECT * FROM local.db.people.files").show()
 
spark.sql("select file_path from local.db.people.files").show(truncate=False)


root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- _change_type: string (nullable = false)
 |-- _change_ordinal: integer (nullable = true)
 |-- _commit_snapshot_id: long (nullable = true)

+-------+--------------------+-----------+-------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+------------+--------------------+--------------+---------------------+--------------------+
|content|           file_path|file_format|spec_id|record_count|file_size_in_bytes|        column_sizes|        value_counts|   null_value_counts|nan_value_counts|        lower_bounds|        upper_bounds|key_metadata|split_offsets|equality_ids|sort_order_id|first_row_id|referenced_data_file|content_offset|content_size_in_bytes|    readable_metrics|
+------

In [27]:
spark.read.format("iceberg").load("local.db.people.partitions").printSchema()
 
spark.sql("SELECT * FROM local.db.people.partitions").show()


root
 |-- record_count: long (nullable = false)
 |-- file_count: integer (nullable = false)
 |-- total_data_file_size_in_bytes: long (nullable = false)
 |-- position_delete_record_count: long (nullable = false)
 |-- position_delete_file_count: integer (nullable = false)
 |-- equality_delete_record_count: long (nullable = false)
 |-- equality_delete_file_count: integer (nullable = false)
 |-- last_updated_at: timestamp (nullable = true)
 |-- last_updated_snapshot_id: long (nullable = true)

+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+--------------------+------------------------+
|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|     last_updated_at|last_updated_snapshot_id|
+------------+----------+-----------------------------+----------------------

In [28]:
spark.sql('''CREATE OR REPLACE TABLE local.db.people1 (
    id INT,
    name STRING,
    city STRING
) 
USING ICEBERG
PARTITIONED BY (city)''')
 
 
spark.read.format("iceberg").load("local.db.people1.partitions").printSchema()

root
 |-- partition: struct (nullable = false)
 |    |-- city: string (nullable = true)
 |-- spec_id: integer (nullable = false)
 |-- record_count: long (nullable = false)
 |-- file_count: integer (nullable = false)
 |-- total_data_file_size_in_bytes: long (nullable = false)
 |-- position_delete_record_count: long (nullable = false)
 |-- position_delete_file_count: integer (nullable = false)
 |-- equality_delete_record_count: long (nullable = false)
 |-- equality_delete_file_count: integer (nullable = false)
 |-- last_updated_at: timestamp (nullable = true)
 |-- last_updated_snapshot_id: long (nullable = true)

